____
__Universidad de San Andrés__<br/>
__Visión Artificial__<br/>
__Sistema de Detección de Infracciones y Estimación de Velocidades__<br/>
__Martín Bianchi, Agustín Amblard y Federico Gutman__
____

### Importamos las librerías necesarias

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import random
import shutil
import yaml
import cv2
import os
from src.aux import *
from src.visualization import *

### Cargamos las rutas y generamos los archivos necesarios 

In [3]:
dataset_root = Path("DETRAC_SPLIT")
yolo_runs_root = Path("yolo_runs")

images_train = dataset_root / "images" / "train" # solo para llenar el campo de la función
test_images_path = dataset_root / "images" / "test"
test_labels_path = dataset_root / "labels" / "test"

trained_yolo_weights = str(yolo_runs_root / "DETRAC_YOLO" / "weights" / "best.pt")

LABELS = {
    0: 'others',
    1: 'car',
    2: 'van',
    3: 'bus'
}

test_yaml_path = create_yaml(
    out_dir=yolo_runs_root, 
    images_train_dest=images_train, 
    images_val_dest=test_images_path,
    labels_dict=LABELS
)

Generando YAML:
    -> Clases (4): {0: 'others', 1: 'car', 2: 'van', 3: 'bus'}
Archivo creado exitosamente en: yolo_runs/data.yaml


### Cargamos el modelo

In [4]:
model = YOLO(trained_yolo_weights)

### Evaluamos su rendimiento en el conjunto de prueba

In [5]:
metrics = model.val(
    data=str(test_yaml_path),
    split='val',
    project=str(yolo_runs_root),
    name='test_results'
)

print("\nResultados de la Evaluación:")
print(f"mAP@50:    {metrics.box.map50:.3f}") 
print(f"mAP@50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

Ultralytics 8.3.234 🚀 Python-3.10.17 torch-2.9.1+cu128 CUDA:0 (NVIDIA L4, 22478MiB)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 1.0±0.2 ms, read: 60.3±33.6 MB/s, size: 72.7 KB)
val: Scanning /home/fedegutman/VisionTrafficGuard/tracking/DETRAC_SPLIT/labels/test... 16123 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 16135/16135 835.8it/s 19.3s<0.0s
val: New cache created: /home/fedegutman/VisionTrafficGuard/tracking/DETRAC_SPLIT/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1009/1009 9.5it/s 1:466<0.1s
                   all      16135     157299      0.858      0.778      0.863      0.731
                others       3383       3430      0.857      0.787      0.878      0.737
                   car      16001     128225      0.881      0.704      0.819      0.664
                   van       7604      13522      0.801      0.734  